In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("=== TAHAP 1: KNOWLEDGE BASE GENERATION ===")
# 1. GENERATE DUMMY CORPUS (Mensimulasikan Modul Akademik Fakultas)
modul_akademik = [
    "Machine Learning adalah cabang dari Kecerdasan Buatan yang berfokus pada penggunaan algoritma untuk meniru cara manusia belajar.",
    "Overfitting terjadi ketika model mengingat data latih dengan sangat baik tetapi gagal memprediksi data uji yang belum pernah dilihat sebelumnya.",
    "Arsitektur API-first memastikan bahwa pengembangan backend dan frontend dapat dilakukan secara paralel oleh tim yang berbeda.",
    "Hukum Gestalt dalam Interaksi Manusia dan Komputer menyatakan bahwa manusia secara alami mengelompokkan elemen visual yang berdekatan.",
    "High Performance Computing memanfaatkan komputasi paralel pada multi-core CPU atau GPU untuk memangkas waktu pemrosesan big data.",
    "Data numerik yang hilang (missing values) umumnya diatasi dengan metode imputasi menggunakan mean atau median.",
    "Convolutional Neural Network (CNN) adalah arsitektur deep learning yang sangat efektif untuk klasifikasi citra digital.",
    "Pembuatan indeks pada basis data relasional (SQL) secara drastis mempercepat proses pencarian data (Query) berskala besar.",
    "Blind spot dalam analisis geospasial mengacu pada area permukiman yang tidak tercakup dalam radius pelayanan fasilitas umum.",
    "Extraneous load adalah beban kognitif berlebih yang dialami pengguna akibat desain antarmuka yang terlalu rumit dan penuh chartjunk."
]

print(f"Total Dokumen di Basis Pengetahuan : {len(modul_akademik)} Modul terindeks.")

print("\n=== TAHAP 2: TEXT VECTORIZATION & RETRIEVAL ENGINE ===")
# 2. INISIALISASI MESIN PENCARI (TF-IDF merepresentasikan pencarian Hybrid Leksikal sederhana)
vectorizer = TfidfVectorizer()
vektor_corpus = vectorizer.fit_transform(modul_akademik)

def cari_referensi(pertanyaan_mahasiswa, threshold_relevansi=0.15):
    """
    Fungsi mesin pencari RAG. Menerima pertanyaan, mencari referensi tertinggi, 
    dan menerapkan ambang batas (threshold) untuk mencegah halusinasi.
    """
    # Mengubah pertanyaan menjadi vektor ruang multidimensi
    vektor_tanya = vectorizer.transform([pertanyaan_mahasiswa])
    
    # Menghitung jarak (kemiripan makna) antara pertanyaan dan seluruh modul
    skor_kemiripan = cosine_similarity(vektor_tanya, vektor_corpus).flatten()
    
    # Mengambil dokumen dengan skor tertinggi
    idx_terbaik = np.argmax(skor_kemiripan)
    skor_tertinggi = skor_kemiripan[idx_terbaik]
    
    # Evaluasi Ambang Batas (Grounding Logic)
    if skor_tertinggi >= threshold_relevansi:
        status = "IN-DOMAIN (Referensi Valid)"
        konteks = modul_akademik[idx_terbaik]
    else:
        status = "OUT-OF-DOMAIN (Pencegahan Halusinasi Aktif)"
        konteks = "[SISTEM MENOLAK]: Materi tidak ditemukan di silabus fakultas."
        
    return skor_tertinggi, status, konteks

print("Mesin pencari RAG (TF-IDF Vector Space Model) berhasil dilatih.")

print("\n=== TAHAP 3: EVALUASI & SIMULASI TANYA JAWAB AI TUTOR ===")
# 3. MENSIMULASIKAN PERTANYAAN (Termasuk pertanyaan jebakan/Out of Domain)
skenario_pengujian = [
    "Apa penyebab utama model gagal saat diuji dengan data baru?", # Benar: Overfitting
    "Bagaimana cara memangkas waktu pemrosesan data yang sangat besar?", # Benar: HPC
    "Jelaskan prinsip Gestalt dalam pembuatan UI/UX!", # Benar: Hukum Gestalt
    "Bagaimana resep dan cara memasak rendang daging yang empuk?", # JEBAKAN (Harus ditolak)
    "Siapa pemenang pemilihan presiden tahun 2024?" # JEBAKAN (Harus ditolak)
]

laporan_evaluasi = []

for idx, q in enumerate(skenario_pengujian):
    print(f"\n[{idx+1}] 👨‍🎓 Mahasiswa Bertanya : '{q}'")
    skor, status, rujukan = cari_referensi(q)
    
    print(f"    🤖 Status Retriever  : {status}")
    print(f"    📊 Skor Cosine       : {skor:.4f}")
    print(f"    📚 Konteks Terambil  : {rujukan}")
    
    laporan_evaluasi.append({
        'Pertanyaan': q,
        'Status': status,
        'Skor Relevansi': round(skor, 4)
    })

print("\n=== KESIMPULAN ARSITEKTUR ===")
# Menampilkan ringkasan sebagai dataframe laporan
df_laporan = pd.DataFrame(laporan_evaluasi)
display(df_laporan) # Hanya berlaku di Colab/Jupyter

sukses_ditolak = len(df_laporan[df_laporan['Status'] == 'OUT-OF-DOMAIN (Pencegahan Halusinasi Aktif)'])
print(f"\n[METRIK] Sistem berhasil memblokir {sukses_ditolak} pertanyaan di luar domain (100% Akurasi Segregasi).")
print("[METRIK] RAG Pipeline siap diintegrasikan dengan modul LLM (IndoBERT/OpenAI) sebagai generator kalimat.")